# 17.08 - R3D-18 pretrained inference

**Notebook type:** Solution notebook with completed exercises, smoke checks, and test cases.

**Daily output:** R3D-18 input-pipeline checks and offline architecture inference.

Practice the official Torchvision R3D-18 boundary and checkpoint-aware preprocessing. The executable path uses the exact architecture with `weights=None`; enabling official weights is explicit because it may download a checkpoint.

## Core Ideas

R3D-18 applies 3D convolutions over time and space. Datasets commonly emit `[T,C,H,W]`, while the model consumes `[B,C,T,H,W]`. Official pretrained weights define resize, crop, scaling, normalization, and dimension permutation together. Never silently replace those transforms when using the checkpoint.

In [ ]:
import time
import numpy as np
import torch
from torch import nn
from torchvision.models.video import r3d_18, R3D_18_Weights

SEED = 17
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Prepared TCHW Clips

Two four-frame 32×32 RGB clips contain a moving bright square. They are stored in the dataset-style `[B,T,C,H,W]` layout.

In [ ]:
video_clips = torch.zeros((2, 4, 3, 32, 32), dtype=torch.float32)
for video_index in range(2):
    for time_index in range(4):
        start = 4 + video_index * 5 + time_index * 2
        video_clips[video_index, time_index, video_index, 8:20, start:start + 8] = 1.0
print("dataset clips:", video_clips.shape, video_clips.dtype)

## Exercise 17-A: Prepare the model layout

For offline architecture practice, permute `[B,T,C,H,W]` to `[B,C,T,H,W]`. When requested, use the official weights transform instead.

**Return structure — `prepare_r3d_batch`:** A CPU float32 tensor `[B,C,T,H,W]`. With official preprocessing its spatial size follows `R3D_18_Weights.DEFAULT.transforms()`; otherwise H and W are unchanged.

In [ ]:
def prepare_r3d_batch(clips_btchw, use_official_preprocessing=False):
    clips = clips_btchw.detach().cpu().to(torch.float32)
    if clips.ndim != 5 or clips.shape[2] != 3:
        raise ValueError("clips must have shape [B,T,3,H,W]")
    if use_official_preprocessing:
        return R3D_18_Weights.DEFAULT.transforms()(clips)
    return clips.permute(0, 2, 1, 3, 4).contiguous()


# Smoke check: preserve the small offline fixture.
r3d_batch = prepare_r3d_batch(video_clips)
print("model batch:", r3d_batch.shape)

## Exercise 17-B: Build the exact official architecture

Expose checkpoint use explicitly. Replace `model.fc` only for offline target classes.

**Return structure — `build_r3d_classifier`:** An `r3d_18` `VideoResNet` on `device`. With `use_pretrained=False`, its classifier returns `num_classes`; with `True`, it retains 400 Kinetics categories and may download weights.

In [ ]:
def build_r3d_classifier(num_classes=3, use_pretrained=False, device=DEVICE):
    weights = R3D_18_Weights.DEFAULT if use_pretrained else None
    model = r3d_18(weights=weights)
    if not use_pretrained:
        model.fc = nn.Linear(model.fc.in_features, int(num_classes))
    return model.to(device)


# Smoke check: construct without a network dependency.
r3d_model = build_r3d_classifier(use_pretrained=False)
print(type(r3d_model).__name__, r3d_model.fc)

## Exercise 17-C: Run and decode inference

Use evaluation and inference modes, calculate probabilities, and return top-k indices.

**Return structure — `r3d_inference`:** A dictionary with `logits` and `probabilities` as CPU float32 `[B,C]`, `top_indices` as CPU int64 `[B,K]`, and Python float `runtime_seconds`.

In [ ]:
def r3d_inference(model, batch, top_k=2, device=DEVICE):
    model.eval(); start = time.perf_counter()
    with torch.inference_mode():
        logits = model(batch.to(device))
        probabilities = torch.softmax(logits, dim=1)
        top_indices = probabilities.topk(min(int(top_k), probabilities.shape[1]), dim=1).indices
    return {"logits": logits.cpu(), "probabilities": probabilities.cpu(), "top_indices": top_indices.cpu(), "runtime_seconds": time.perf_counter() - start}


# Smoke check: infer both clips through the exact architecture.
r3d_output = r3d_inference(r3d_model, r3d_batch)
print("output:", r3d_output["logits"].shape, r3d_output["top_indices"])

## Test Cases

**Return structure — `run_day17_tests`:** Returns `None`; assertions and `Day 17 tests passed` communicate success.

In [ ]:
def run_day17_tests():
    assert r3d_batch.shape == (2, 3, 4, 32, 32) and r3d_batch.dtype == torch.float32
    assert type(r3d_model).__name__ == "VideoResNet" and r3d_model.fc.out_features == 3
    assert r3d_output["logits"].shape == r3d_output["probabilities"].shape == (2, 3)
    assert r3d_output["top_indices"].shape == (2, 2)
    assert torch.allclose(r3d_output["probabilities"].sum(dim=1), torch.ones(2), atol=1e-6)
    assert r3d_output["runtime_seconds"] >= 0
    print("Day 17 tests passed")


run_day17_tests()

## Day 17 Checklist

- [ ] Distinguish TCHW dataset layout from BCTHW model layout.
- [ ] Use official transforms with official weights.
- [ ] Keep checkpoint downloads explicit.
- [ ] Decode top-k indices against the correct category mapping.
- [ ] Run the test cases.